# Functional Programming & Stream API in Java (Java 8+)
Java 8 introduced functional programming capabilities, drastically reducing boilerplate and enabling declarative, parallel pipeline processing.

### Key Components:
1. **Functional Interface**: An interface with exactly **one abstract method** (`@FunctionalInterface`).
2. **Built-in Functional Interfaces (`java.util.function`)**:
   - `Predicate<T>`: Evaluates condition, returns `boolean` (`test(T)`).
   - `Function<T, R>`: Transforms input `T` into output `R` (`apply(T)`).
   - `Consumer<T>`: Consumes input, returns `void` (`accept(T)`).
   - `Supplier<T>`: Takes no input, produces output `T` (`get()`).
3. **Lambda Expressions**: `(parameters) -> expression`
4. **Method References (`::`)**: `ClassName::staticMethod`, `obj::instanceMethod`, `ClassName::new`.
5. **Stream API**: Sequence of elements supporting functional data processing queries.


---
### Benefits and Real-World Applications of Lambda Expressions (Topic 108)
1. **Dramatically Reduces Boilerplate**: Replaces 5-6 lines of verbose anonymous inner class declarations with concise single-line expressions.
2. **Enables Functional Programming**: Allows treating algorithms, functions, and behaviors as first-class citizens that can be stored in variables or passed as method parameters.
3. **Powers the Stream API**: Provides the foundational building blocks for functional declarative pipelines (`filter`, `map`, `reduce`).
4. **Facilitates Lazy Evaluation**: Computations defined via Lambdas and Streams are deferred until the result is strictly demanded by a terminal operation.
5. **Effortless Concurrency**: Enables parallel execution via `parallelStream()` without requiring developers to write complex locking or thread management code.


---
## 1. Built-in Functional Interfaces and Method References


In [ ]:
import java.util.function.Predicate;
import java.util.function.Function;
import java.util.function.Consumer;
import java.util.function.Supplier;

// 1. Predicate: test condition
Predicate<Integer> isAdult = age -> age >= 18;
System.out.println("Age 21 isAdult: " + isAdult.test(21));

// 2. Function: transform string to length
Function<String, Integer> stringLength = String::length; // Method reference!
System.out.println("Length of 'CDAC': " + stringLength.apply("CDAC"));

// 3. Consumer: print value
Consumer<String> printer = System.out::println; // Method reference
printer.accept("Printed using Consumer & Method Reference!");

// 4. Supplier: generate random double
Supplier<Double> randomSupplier = Math::random;
System.out.println("Generated Random Value: " + randomSupplier.get());


In [ ]:
import java.util.function.Function;
import java.util.function.Supplier;
import java.util.ArrayList;
import java.util.List;

class Product {
    String productName;
    public Product(String name) {
        this.productName = name;
    }
    @Override
    public String toString() { return "Product: " + productName; }
}

// 1. Constructor Reference with Argument: ClassName::new
Function<String, Product> productFactory = Product::new;
Product p1 = productFactory.apply("Mechanical Keyboard");
System.out.println("Constructed via Method Reference: " + p1);

// 2. Constructor Reference with Zero Arguments: ClassName::new
Supplier<List<String>> listGenerator = ArrayList::new;
List<String> dynamicList = listGenerator.get();
dynamicList.add("Instantiated via Supplier Constructor Reference");
System.out.println(dynamicList.get(0));


---
## 2. Stream API: Intermediate and Terminal Operations
- **Intermediate**: `filter()`, `map()`, `flatMap()`, `sorted()`, `distinct()`, `limit()`, `skip()`
- **Terminal**: `forEach()`, `collect()`, `count()`, `reduce()`, `min()`, `max()`, `findFirst()`, `anyMatch()`, `allMatch()`, `noneMatch()`


In [ ]:
import java.util.Arrays;
import java.util.List;
import java.util.Optional;
import java.util.stream.Collectors;

List<Integer> nums = Arrays.asList(5, 2, 8, 1, 9, 2, 4, 8, 3);

// 1. Intermediate operations: filter, distinct, sorted, limit
List<Integer> processed = nums.stream()
    .filter(n -> n > 2)
    .distinct()
    .sorted()
    .limit(4)
    .collect(Collectors.toList());
System.out.println("Processed Stream List: " + processed);

// 2. Terminal operations: count, reduce
long count = nums.stream().filter(n -> n % 2 == 0).count();
int sum = nums.stream().reduce(0, (a, b) -> a + b);
System.out.println("Count of even numbers: " + count);
System.out.println("Sum of numbers: " + sum);

// 3. Terminal operations: min(), max(), findFirst()
Optional<Integer> minVal = nums.stream().min(Integer::compareTo);
Optional<Integer> maxVal = nums.stream().max(Integer::compareTo);
Optional<Integer> firstVal = nums.stream().filter(n -> n > 5).findFirst();
System.out.println("Min Value   : " + minVal.orElse(0));
System.out.println("Max Value   : " + maxVal.orElse(0));
System.out.println("First > 5   : " + firstVal.orElse(0));

// 4. Matching: anyMatch, allMatch, noneMatch
boolean hasEven = nums.stream().anyMatch(n -> n % 2 == 0);
boolean allPositive = nums.stream().allMatch(n -> n > 0);
System.out.println("Has even? " + hasEven + " | All positive? " + allPositive);


---
### Intermediate Operations Deep Dive: `flatMap()` and `skip()` (Topic 110)
- **`flatMap()`**: Transforms each element into a stream of values and flattens multiple nested streams into a single combined one-dimensional stream ($O(N \times M) \rightarrow O(N)$).
- **`skip(n)`**: Discards the first $n$ elements of the stream, emitting only the remainder (essential for database-style pagination).


In [ ]:
import java.util.Arrays;
import java.util.List;
import java.util.stream.Collectors;

// 1. flatMap Demonstration: Flattening nested lists
List<List<String>> nestedTeams = Arrays.asList(
    Arrays.asList("Alice", "Bob"),
    Arrays.asList("Charlie", "Diana"),
    Arrays.asList("Edward", "Fiona")
);

List<String> allMembers = nestedTeams.stream()
    .flatMap(List::stream) // Flattens each inner list stream into one unified stream
    .collect(Collectors.toList());

System.out.println("Flattened All Members: " + allMembers);

// 2. skip() and limit() Demonstration: Pagination (Page 2 with page size 3)
List<Integer> dataset = Arrays.asList(10, 20, 30, 40, 50, 60, 70, 80, 90);
int pageNumber = 2; // 1-indexed
int pageSize = 3;

List<Integer> page2Items = dataset.stream()
    .skip((pageNumber - 1) * pageSize) // skip first 3 items (10, 20, 30)
    .limit(pageSize)                   // take next 3 items (40, 50, 60)
    .collect(Collectors.toList());

System.out.println("Pagination Page 2 elements (skip & limit): " + page2Items);


---
## 3. `Collectors` Utility Class & Grouping
Powerful reductions like `groupingBy()`, `partitioningBy()`, and `joining()`:


In [ ]:
import java.util.Map;
import java.util.stream.Collectors;

class EmployeeData {
    String name;
    String dept;
    double salary;

    EmployeeData(String name, String dept, double salary) {
        this.name = name;
        this.dept = dept;
        this.salary = salary;
    }
}

List<EmployeeData> employees = Arrays.asList(
    new EmployeeData("Alice", "IT", 75000),
    new EmployeeData("Bob", "HR", 50000),
    new EmployeeData("Charlie", "IT", 90000),
    new EmployeeData("Diana", "HR", 60000),
    new EmployeeData("Edward", "Finance", 80000)
);

// 1. Group employees by Department
Map<String, List<EmployeeData>> byDept = employees.stream()
    .collect(Collectors.groupingBy(e -> e.dept));

System.out.println("Employees Grouped by Department:");
byDept.forEach((dept, empList) -> {
    System.out.println("Department: " + dept + " (" + empList.size() + " employees)");
});

// 2. Joining names into a comma-separated string
String allNames = employees.stream()
    .map(e -> e.name)
    .collect(Collectors.joining(", "));
System.out.println("\nAll Employee Names: " + allNames);

// 3. Parallel Stream demo (utilizes multi-core processors)
long highEarnerCount = employees.parallelStream()
    .filter(e -> e.salary >= 70000)
    .count();
System.out.println("High Earners count (Parallel Stream): " + highEarnerCount);


---
### Collectors: `toSet()` and `partitioningBy()` (Topic 112)
- **`Collectors.toSet()`**: Collects stream elements into a `Set`, automatically eliminating duplicate values.
- **`Collectors.partitioningBy(Predicate)`**: A specialized form of grouping that splits stream elements into exactly **two partitions** (`true` and `false`) based on a boolean predicate, returning `Map<Boolean, List<T>>`.


In [ ]:
import java.util.Arrays;
import java.util.List;
import java.util.Set;
import java.util.Map;
import java.util.stream.Collectors;

List<Integer> rawNumbers = Arrays.asList(12, 45, 12, 60, 33, 45, 78, 91, 60);

// 1. toSet(): deduplicate into Set
Set<Integer> uniqueNumbers = rawNumbers.stream()
    .collect(Collectors.toSet());
System.out.println("Collected toSet() (Duplicates eliminated): " + uniqueNumbers);

// 2. partitioningBy(): partition into Even (true) vs Odd (false)
Map<Boolean, List<Integer>> partitioned = rawNumbers.stream()
    .collect(Collectors.partitioningBy(n -> n % 2 == 0));

System.out.println("Even Numbers (true) : " + partitioned.get(true));
System.out.println("Odd Numbers  (false): " + partitioned.get(false));


---
## 4. Parallel Streams Deep Dive (Topic 113)
Parallel streams split data across multiple threads using the JVM's underlying **`ForkJoinPool.commonPool()`**:
- **Work-Stealing Algorithm**: Idle threads steal tasks from busy threads to maintain maximum CPU core saturation.
- **When to Use**:
  - Very large datasets ($N > 10,000$ elements).
  - Computationally intensive operations (CPU-bound algorithms).
  - The stream operations are completely **stateless and side-effect free**.
- **When NOT to Use**:
  - Small collections (parallel overhead exceeds gains).
  - I/O bound operations (blocking threads starves the common pool).
  - Operations that depend on order (`findFirst`, `limit`).
  - Operations that mutate shared variables (causes race conditions)!

---
## 5. Advantages of Stream API Over Traditional Iteration (Topic 114)

| Criterion | Traditional Iteration (`for` / `while` loop) | Stream API (`stream()`) |
| :--- | :--- | :--- |
| **Programming Style** | **Imperative** (Explicitly specifies *HOW* to do it: index, increment, bounds) | **Declarative** (Specifies *WHAT* to do: filter, map, collect) |
| **Iteration Type** | **External Iteration** (Client code controls the loop flow) | **Internal Iteration** (JVM controls iteration and optimizes traversal) |
| **Evaluation** | **Eager** (Each line executes immediately) | **Lazy Evaluation** (Intermediate steps run only when terminal operation executes) |
| **Parallelism** | Very difficult (requires manual threads, synchronized locks, executors) | Trivial (`.parallel()` or `parallelStream()` handles thread distribution) |
| **Purity & Mutability** | Frequently mutates external accumulator variables | Encourages immutability and side-effect free functional pipelines |
| **Readability** | High boilerplate; hard to decipher intent in nested loops | Highly readable, self-documenting method chains |
